<a href="https://colab.research.google.com/github/David-NNdungu/econ5200-Problem-set-1-/blob/main/Econ_5200_PS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ECON 5200 — Problem Set 1: The Measurement Audit
**David Ndungu**

I audited a simulated retailer dashboard that reports 8.59% growth in average basket value, while the true consumer average falls by 3.19%. I identified the two measurement problems, compared alternative statistics, and recommended a corrected measure.

**Metric definition:** average basket value is consumer revenue divided by the number of completed consumer orders. Each order receives equal weight, including repeated orders from the same customer. All amounts are dollars, and gaps between growth rates are measured in percentage points.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import os

pd.options.display.float_format = "{:,.4f}".format
SEED = 2106432505

## Phase 1: Reproduce the Wrong Number
### Step 1.1: Build the transaction data
I simulated 11,183 completed consumer orders in 2024 and 12,438 in 2025 using right-skewed lognormal distributions. I used a fixed random seed so I could reproduce the results. Each order has a unique ID; customers can place multiple orders.

I recorded each year's true average and saved a finance ledger before adding either fault.

In [2]:
rng = np.random.default_rng(SEED)
# Random parameters and counts; the fixed seed makes this draw reproducible.
consumer_counts = {2024: int(rng.integers(10500, 12501)),
                   2025: int(rng.integers(11000, 13501))}
log_mean = float(rng.uniform(3.6, 3.9))
log_sigma = float(rng.uniform(0.65, 0.85))
parts = []
for year in [2024, 2025]:
    n = consumer_counts[year]
    year_mean = log_mean
    if year == 2025:
        year_mean = log_mean - 0.035
    amounts = rng.lognormal(mean=year_mean,
                            sigma=log_sigma, size=n)
    parts.append(pd.DataFrame({
        "order_id": [f"C-{year}-{i}" for i in range(n)],
        "customer_id": rng.integers(1, 5001, size=n),
        "year": year, "basket_value": amounts,
        "customer_type": "consumer", "status": "completed"
    }))
consumer_truth = pd.concat(parts, ignore_index=True)
print(f"Created {len(consumer_truth):,} clean completed consumer orders.")

Created 23,621 clean completed consumer orders.


In [3]:
# Record the truth before contaminating the transactions.
truth_means = consumer_truth.groupby("year").basket_value.mean().copy()
finance_ledger = consumer_truth[["order_id", "year", "basket_value"]].copy()
finance_ledger = finance_ledger.rename(columns={"basket_value": "recognized_revenue"})
print("Truth recorded before contamination:")
display(truth_means.to_frame("true_consumer_average"))
assert len(consumer_truth) >= 20000
assert consumer_truth.order_id.is_unique

Truth recorded before contamination:


,true_consumer_average
year,
2024,61.5170
2025,59.5535


### Plant the two faults
I added 20 large B2B orders in 2024 and 94 in 2025. I also added 858 cancelled consumer orders to the 2025 log, with an average intended basket value of $115.91 and no recognized revenue. The earlier log excluded cancellations.

In [4]:
# Large basket values for the simulated B2B orders.
TAIL_TEMPLATE = [1268.655029296875, 1064.0, 910.02001953125, 873.905029296875, 865.7949829101562, 833.0, 794.9957885742188, 794.2899780273438, 781.0, 753.5, 732.0, 729.75, 725.0999755859375, 720.0133056640625, 718.4400024414062, 718.0, 713.4500122070312, 702.0, 699.243896484375, 677.2149658203125]
b2b_counts = {2024: int(rng.integers(18, 36)), 2025: int(rng.integers(65, 101))}
n_cancel = int(rng.integers(450, 901))
cancel_log_mean = float(rng.uniform(4.55, 4.95))
cancel_values = rng.lognormal(cancel_log_mean, 0.5, n_cancel)
b2b_parts = []
for year, n_b2b in b2b_counts.items():
    amounts = rng.choice(TAIL_TEMPLATE, size=n_b2b, replace=True)
    # Small random variation avoids copying a fixed set of amounts verbatim.
    amounts *= rng.uniform(0.95, 1.05, size=n_b2b)
    b2b_parts.append(pd.DataFrame({
        "order_id": [f"B-{year}-{i}" for i in range(n_b2b)],
        "customer_id": np.arange(6001, 6001+n_b2b), "year": year,
        "basket_value": amounts, "customer_type": "B2B", "status": "completed"}))
print("B2B mean by year:")
display(pd.concat(b2b_parts).groupby("year").basket_value.mean().to_frame("B2B_mean"))

B2B mean by year:


,B2B_mean
year,
2024,794.5489
2025,779.9530


In [5]:
# Only the second year's log includes cancelled consumer orders.
cancelled = pd.DataFrame({
    "order_id": [f"X-2025-{i}" for i in range(n_cancel)],
    "customer_id": rng.integers(1, 5001, n_cancel), "year": 2025,
    "basket_value": cancel_values, "customer_type": "consumer", "status": "cancelled"})
transactions = pd.concat([consumer_truth, b2b_parts[0], b2b_parts[1], cancelled], ignore_index=True)
display(pd.crosstab(transactions.year, [transactions.customer_type, transactions.status]))
print(f"Total logged rows: {len(transactions):,}")

customer_type       B2B  consumer          
status        completed cancelled completed
year                                       
2024                 20         0     11183
2025                 94       858     12438

Total logged rows: 24,593


### Step 1.2: Reproduce the dashboard
The dashboard takes the mean of every logged row, including B2B and cancelled orders. I compared that result to the recorded truth using the same growth formula in both cases.

In [6]:
naive_means = transactions.groupby("year")["basket_value"].mean()
true_growth = (truth_means.loc[2025] / truth_means.loc[2024] - 1) * 100
dashboard_growth = (naive_means.loc[2025] / naive_means.loc[2024] - 1) * 100
gap = dashboard_growth - true_growth

comparison = pd.DataFrame({"true_average": truth_means, "dashboard_average": naive_means})
comparison["overstatement_dollars"] = comparison["dashboard_average"] - comparison["true_average"]
display(comparison)
print(f"Dashboard growth: {dashboard_growth:.2f}%")
print(f"True growth: {true_growth:.2f}%")
print(f"Growth overstatement: {gap:.2f} percentage points")
assert dashboard_growth > 0
assert true_growth < 0

,true_average,dashboard_average,overstatement_dollars
year,,,
2024,61.5170,62.8257,1.3086
2025,59.5535,68.2219,8.6684


Dashboard growth: 8.59%
True growth: -3.19%
Growth overstatement: 11.78 percentage points


I found that the true average fell from $61.52 to $59.55, a 3.19% decline. The dashboard reported 8.59% growth, a gap of 11.78 percentage points.

## Phase 2: Locate the Cause
### Step 2.1: Separate the two mechanisms
I recalculated the mean with only B2B orders removed, only cancellations removed, and both removed. Each repair changes the denominator, so its contribution depends on the repair order. I calculated both orders and averaged their contributions.

In [7]:
# Repair B2B only, logging only, and then both.
no_b2b = transactions[transactions["customer_type"] == "consumer"]
completed = transactions[transactions["status"] == "completed"]
clean_orders = transactions[(transactions["customer_type"] == "consumer") &
                            (transactions["status"] == "completed")]

no_b2b_means = no_b2b.groupby("year")["basket_value"].mean()
completed_means = completed.groupby("year")["basket_value"].mean()
clean_means = clean_orders.groupby("year")["basket_value"].mean()

scenario_means = pd.DataFrame({
    "Both faults": naive_means,
    "B2B repaired only": no_b2b_means,
    "Logging repaired only": completed_means,
    "Both repaired": clean_means
})
scenario_growth = (scenario_means.loc[2025] / scenario_means.loc[2024] - 1) * 100
display(scenario_means)
display(scenario_growth.to_frame("growth_pct"))

,Both faults,B2B repaired only,Logging repaired only,Both repaired
year,,,,
2024,62.8257,61.5170,62.8257,61.5170
2025,68.2219,63.1901,64.9570,59.5535


,growth_pct
Both faults,8.5893
B2B repaired only,2.7197
Logging repaired only,3.3925
Both repaired,-3.1919


In [8]:
# B2B first: change in growth after excluding B2B, then cancellations.
raw_growth = scenario_growth["Both faults"]
b2b_repaired_growth = scenario_growth["B2B repaired only"]
logging_repaired_growth = scenario_growth["Logging repaired only"]
clean_growth = scenario_growth["Both repaired"]

b2b_first = raw_growth - b2b_repaired_growth
logging_second = b2b_repaired_growth - clean_growth

# Logging first: repeat the calculation in the other order.
logging_first = raw_growth - logging_repaired_growth
b2b_second = logging_repaired_growth - clean_growth

# Average the two orders so our attribution does not depend on repair order.
b2b_contribution = (b2b_first + b2b_second) / 2
logging_contribution = (logging_first + logging_second) / 2

decomposition = pd.DataFrame({
    "repair_order": ["B2B first", "Logging first", "Average of both orders"],
    "B2B_contribution_pp": [b2b_first, b2b_second, b2b_contribution],
    "logging_contribution_pp": [logging_second, logging_first, logging_contribution]
})
decomposition["total_pp"] = decomposition["B2B_contribution_pp"] + decomposition["logging_contribution_pp"]
display(decomposition)
assert np.allclose(decomposition["total_pp"], gap)

# The two repairs also explain each year's dollar overstatement.
levels = pd.DataFrame({
    "B2B_dollars": naive_means - no_b2b_means,
    "logging_dollars": no_b2b_means - clean_means
})
levels["total_dollars"] = levels["B2B_dollars"] + levels["logging_dollars"]
display(levels)
assert np.allclose(levels["total_dollars"], comparison["overstatement_dollars"])

,repair_order,B2B_contribution_pp,logging_contribution_pp,total_pp
0,B2B first,5.8695,5.9116,11.7811
1,Logging first,6.5844,5.1967,11.7811
2,Average of both orders,6.2270,5.5542,11.7811


,B2B_dollars,logging_dollars,total_dollars
year,,,
2024,1.3086,0.0000,1.3086
2025,5.0318,3.6367,8.6684


| Mechanism | Contribution to the growth gap |
|---|---:|
| B2B tail | 6.2270 pp |
| Newly logged cancellations | 5.5542 pp |
| Total | 11.7811 pp |

I averaged the two repair orders to split their interaction equally. Right skew among valid consumer orders is not itself a measurement error.

### Step 2.2: Robust alternatives
I calculated the median, a mean trimmed by 10% at each end, and the mean after excluding B2B accounts. I also calculated these statistics on completed consumer orders so both years used the same inclusion rule.

In [9]:
rows = []
for year in [2024, 2025]:
    logged = transactions[transactions["year"] == year]
    consumer = no_b2b[no_b2b["year"] == year]
    clean = clean_orders[clean_orders["year"] == year]
    rows.append({
        "year": year,
        "logged_median": logged["basket_value"].median(),
        "logged_trimmed_10pct": stats.trim_mean(logged["basket_value"], 0.10),
        "B2B_excluded_only": consumer["basket_value"].mean(),
        "comparable_median": clean["basket_value"].median(),
        "comparable_trimmed_10pct": stats.trim_mean(clean["basket_value"], 0.10),
        "completed_consumer_mean": clean["basket_value"].mean()
    })

alternatives = pd.DataFrame(rows).set_index("year")
display(alternatives.T)
alternative_growth = (alternatives.loc[2025] / alternatives.loc[2024] - 1) * 100
display(alternative_growth.to_frame("growth_pct"))

year,2024,2025
logged_median,45.4859,46.9939
logged_trimmed_10pct,52.0972,53.8212
B2B_excluded_only,61.5170,63.1901
comparable_median,45.4146,44.0394
comparable_trimmed_10pct,51.9455,49.8267
completed_consumer_mean,61.5170,59.5535


,growth_pct
logged_median,3.3152
logged_trimmed_10pct,3.3092
B2B_excluded_only,2.7197
comparable_median,-3.0282
comparable_trimmed_10pct,-4.0790
completed_consumer_mean,-3.1919


**2025 values on the logged population:** median $46.99; 10% trimmed mean $53.82; mean after B2B exclusion $63.19. The last number still includes cancelled orders. The results above also show each estimator on the comparable completed-consumer population.

| Alternative | Assumption that makes it appropriate | When it is wrong |
|---|---|---|
| Median | The client wants the middle completed consumer order, with half above and half below. | It is used as revenue per order; it ignores the size of the upper tail. |
| 10% trimmed mean from each tail | The client wants a central-basket indicator and accepts dropping 20% of orders. | Large and small orders are valid and their revenue belongs in the average. |
| Arithmetic mean after explicit B2B exclusion | Account classification is accurate and the target is consumer spending per order. | Cancelled orders are still included, account types are misclassified, or the client actually wants a customer-level statistic. |

All three require the same completed-order logging rule in both years. A price cutoff would also risk deleting genuine large consumer purchases.

**Provisional recommendation before the AI review:** show the 10% trimmed mean for completed consumers after explicit B2B exclusion. It reduces the influence of extreme basket values. I tested whether that recommendation fits finance's definition in Phase 4 before making the final decision.

## Phase 3: Ship the Fix
### Step 3.1: The module
I wrote `src/basket_metrics.py` with `audit_report` and `robust_mean`. The audit reports missingness, dtype, skew, and outliers using the 1.5-IQR rule. An outlier flag identifies an unusual value; it does not prove an error.

In [10]:
os.makedirs("src", exist_ok=True)

In [11]:
%%writefile src/basket_metrics.py
"""Simple audit checks and basket statistics."""

import numpy as np
import pandas as pd
from scipy import stats


def audit_report(df: pd.DataFrame) -> pd.DataFrame:
    """Return missingness, dtype, skew and 1.5-IQR outlier counts by column.

    Skew and outlier counts are unavailable for non-numeric columns.
    An outlier flag does not mean that the observation is an error.
    """
    rows = []

    for column in df.columns:
        values = df[column]
        skew = np.nan
        outliers = np.nan

        if pd.api.types.is_numeric_dtype(values) and not pd.api.types.is_bool_dtype(values):
            clean = values.dropna()
            if len(clean) > 0:
                q1 = clean.quantile(0.25)
                q3 = clean.quantile(0.75)
                iqr = q3 - q1
                lower = q1 - 1.5 * iqr
                upper = q3 + 1.5 * iqr
                outliers = ((clean < lower) | (clean > upper)).sum()
                skew = clean.skew()

        rows.append({
            "column": column,
            "missing_count": values.isna().sum(),
            "missing_pct": values.isna().mean() * 100,
            "dtype": str(values.dtype),
            "skew": skew,
            "outlier_count": outliers
        })

    return pd.DataFrame(rows).set_index("column")


def robust_mean(x: pd.Series | list[float], method: str = "median",
                trim_fraction: float = 0.10,
                exclude_mask: pd.Series | list[bool] | None = None) -> float:
    """Calculate a median, trimmed mean or mean after an explicit exclusion.

    trim_fraction is removed from each tail. For method='rule', provide
    a Boolean mask in the same row order as x; True means exclude.
    Missing values are dropped after the rule. Empty data raises an error.
    """
    values = pd.Series(x, dtype=float).reset_index(drop=True)

    if method == "rule":
        if exclude_mask is None:
            raise ValueError("Provide an exclusion mask for method='rule'.")
        mask = pd.Series(exclude_mask).reset_index(drop=True)
        if len(mask) != len(values) or mask.dtype != bool or mask.isna().any():
            raise ValueError("Use a Boolean mask with one value per observation.")
        values = values[~mask]

    values = values.dropna()
    if len(values) == 0:
        raise ValueError("No observations remain.")
    if np.isinf(values).any():
        raise ValueError("Values must be finite.")

    if method == "median":
        return float(values.median())
    elif method == "trimmed":
        if trim_fraction < 0 or trim_fraction >= 0.5:
            raise ValueError("trim_fraction must be between 0 and 0.5.")
        return float(stats.trim_mean(values, trim_fraction))
    elif method == "rule":
        return float(values.mean())
    else:
        raise ValueError("Choose median, trimmed or rule.")


if __name__ == "__main__":
    example = pd.DataFrame({
        "basket_value": [20., 30., 40., 2000., np.nan],
        "customer_type": ["consumer", "consumer", "consumer", "B2B", "consumer"]
    })
    print(audit_report(example))
    print("Median:", robust_mean(example["basket_value"], "median"))
    print("Trimmed mean:", robust_mean(example["basket_value"], "trimmed", 0.25))
    print("Consumer mean:", robust_mean(example["basket_value"], "rule",
          exclude_mask=example["customer_type"] == "B2B"))

Writing src/basket_metrics.py


In [12]:
import sys
import importlib
sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)

display(basket_metrics.audit_report(transactions))

,missing_count,missing_pct,dtype,skew,outlier_count
column,,,,,
order_id,0,0.0000,object,NaN,NaN
customer_id,0,0.0000,int64,0.0335,0.0000
year,0,0.0000,int64,-0.1786,0.0000
basket_value,0,0.0000,float64,5.7560,"1,560.0000"
customer_type,0,0.0000,object,NaN,NaN
status,0,0.0000,object,NaN,NaN


In [13]:
!python src/basket_metrics.py

               missing_count  missing_pct    dtype      skew  outlier_count
column                                                                     
basket_value               1         20.0  float64  1.999588            1.0
customer_type              0          0.0   object       NaN            NaN
Median: 35.0
Trimmed mean: 35.0
Consumer mean: 30.0


### Step 3.2: Verification checkpoint
I tested the module on new data. These seven assertions check symmetric data, trimming, exclusion with missing values, and the audit report.

In [14]:
symmetric = pd.Series([-3., -2., -1., 0., 1., 2., 3.])
assert np.isclose(basket_metrics.robust_mean(symmetric, "median"), symmetric.mean())
assert np.isclose(basket_metrics.robust_mean(symmetric, "trimmed"), symmetric.mean())

extreme = pd.Series([10., 20., 30., 40., 10000.])
assert np.isclose(basket_metrics.robust_mean(extreme, "trimmed", 0.20), 30)

values = pd.Series([10., np.nan, 20., 9000.])
exclude = pd.Series([False, False, False, True])
assert np.isclose(basket_metrics.robust_mean(values, "rule", exclude_mask=exclude), 15)

test_data = pd.DataFrame({"amount": [1., 2., 3., 4., 100., np.nan],
                          "label": ["a", "b", None, "d", "e", "f"]})
report = basket_metrics.audit_report(test_data)
assert report.loc["amount", "missing_count"] == 1
assert report.loc["label", "dtype"] == str(test_data["label"].dtype)
assert report.loc["amount", "outlier_count"] == 1
print("All 7 module assertions passed.")

All 7 module assertions passed.


## Phase 4: AI Expansion
### Task 4.1: The adversarial review
I used the assignment's P.R.I.M.E. prompt to challenge my provisional recommendation. The exact prompt, full response, and change log are in [ai-appendix.md](ai-appendix.md).

The strongest objection was that trimming valid consumer orders changes the metric. I tested whether my recommendation matched recognized consumer revenue divided by completed consumer orders.

### Test the strongest objection
I matched the completed consumer orders to the finance ledger I saved before contamination. I checked every order ID, year, and amount, then compared revenue per order with the trimmed mean. I also calculated how many valid orders and how much revenue trimming removed.

In [15]:
# Match every included order against the ledger saved before contamination.
reconciled = clean_orders.merge(finance_ledger, on="order_id", how="outer",
                                suffixes=("_orders", "_finance"), indicator=True,
                                validate="one_to_one")
assert (reconciled["_merge"] == "both").all()
assert (reconciled["year_orders"] == reconciled["year_finance"]).all()
assert np.allclose(reconciled["basket_value"], reconciled["recognized_revenue"])

revenue = finance_ledger.groupby("year")["recognized_revenue"].sum()
order_count = finance_ledger.groupby("year")["order_id"].count()
finance_average = revenue / order_count

reconciliation = pd.DataFrame({
    "revenue": revenue,
    "completed_orders": order_count,
    "revenue_per_order": finance_average,
    "corrected_mean": clean_means,
    "proposed_trimmed_mean": alternatives["comparable_trimmed_10pct"]
})
reconciliation["trimmed_minus_finance"] = reconciliation["proposed_trimmed_mean"] - reconciliation["revenue_per_order"]
display(reconciliation)
assert np.allclose(reconciliation["corrected_mean"], reconciliation["revenue_per_order"])

,revenue,completed_orders,revenue_per_order,corrected_mean,proposed_trimmed_mean,trimmed_minus_finance
year,,,,,,
2024,"687,944.8731",11183,61.5170,61.5170,51.9455,-9.5715
2025,"740,726.1367",12438,59.5535,59.5535,49.8267,-9.7268


In [16]:
# Show what the 10% trim removes from real consumer orders in this example.
rows = []
for year in [2024, 2025]:
    orders = clean_orders[clean_orders["year"] == year].sort_values("basket_value")
    trim_count = int(len(orders) * 0.10)
    retained = orders.iloc[trim_count:len(orders)-trim_count]
    removed_revenue = orders["basket_value"].sum() - retained["basket_value"].sum()
    rows.append({"year": year,
                 "removed_orders": len(orders) - len(retained),
                 "removed_revenue": removed_revenue,
                 "removed_revenue_pct": removed_revenue / orders["basket_value"].sum() * 100})
display(pd.DataFrame(rows).set_index("year"))

# Compare trim rates on the same completed consumer population.
values_2024 = clean_orders.loc[clean_orders["year"] == 2024, "basket_value"]
values_2025 = clean_orders.loc[clean_orders["year"] == 2025, "basket_value"]
rows = []
for rate in [0, 0.05, 0.10, 0.20]:
    mean_2024 = stats.trim_mean(values_2024, rate)
    mean_2025 = stats.trim_mean(values_2025, rate)
    change = (mean_2025 / mean_2024 - 1) * 100
    rows.append({"trim_each_tail": rate, "average_2024": mean_2024,
                 "average_2025": mean_2025, "growth_pct": change})
display(pd.DataFrame(rows))

,removed_orders,removed_revenue,removed_revenue_pct
year,,,
2024,2236,"223,188.3252",32.4428
2025,2486,"244,851.0418",33.0555


,trim_each_tail,average_2024,average_2025,growth_pct
0,0.0000,61.5170,59.5535,-3.1919
1,0.0500,54.7262,52.7758,-3.5639
2,0.1000,51.9455,49.8267,-4.0790
3,0.2000,48.6988,46.7060,-4.0921


I found that the corrected arithmetic mean matched the saved finance ledger: $687,944.87 / 11,183 = $61.52 in 2024 and $740,726.14 / 12,438 = $59.55 in 2025. The 10% trimmed means were $51.95 and $49.83. Trimming removed 2,236 valid orders in 2024 and 2,486 in 2025, along with their revenue.

I changed my recommendation to the arithmetic mean of completed consumer orders, excluding B2B accounts and cancellations in both years. The dashboard should show **$59.55 in 2025 and a 3.19% decline**. This matches the metric's revenue-per-order definition.

I have high confidence within this simulation because I recovered the recorded truth. Before using the rule for a real retailer, I would validate order status, account classification, and recognized revenue against independent records.

### Task 4.2: The board slide

**Average basket value: $59.55 — down 3.19%, versus 8.59% reported growth**

**Correction:** use recognized revenue per completed consumer order. Exclude B2B accounts and cancelled orders consistently in both years. Both annual dashboard averages were inflated.

**Cause:** averaging both repair orders attributes 6.23 percentage points to the B2B tail and 5.55 to newly logged cancellations, totaling 11.78 points.

**Confidence:** high within the simulation. All 23,621 eligible orders reconcile to the saved finance ledger. Revenue per completed consumer order matches the recorded truth in both years. The trimmed proposal fails that revenue-per-order check, so I replaced it with the corrected arithmetic mean.

**Before betting on it:** validate the same inclusion rules against the retailer's independent finance and account records.